# DEV 1 — ML Lead | Hackathon FPD TMB
**Objetivo:** Treinar, otimizar e combinar os modelos de ML para prever FPD (First Payment Default).

## FASE 0 — Setup do Ambiente

In [5]:
# ── Imports ──────────────────────────────────────────────────────────────────
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve,
    classification_report
)
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

# Otimização e explicabilidade
import optuna
import shap

# Desbalanceamento
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── SEED GLOBAL — NÃO ALTERAR ────────────────────────────────────────────────
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_DIR    = os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else os.getcwd()
OUTPUTS_DIR = os.path.join(BASE_DIR, 'outputs')
os.makedirs(OUTPUTS_DIR, exist_ok=True)

print('✓ Todas as libs importadas com sucesso!')
print(f'✓ RANDOM_STATE = {RANDOM_STATE}')
print(f'✓ outputs/ em: {OUTPUTS_DIR}')

✓ Todas as libs importadas com sucesso!
✓ RANDOM_STATE = 42
✓ outputs/ em: /home/jonatas/Hackathon/outputs


---
## FASE 1 — Auditoria e Anti-Leakage
> **SUPORTE** — receber a lista de variáveis aprovadas de Dev 2 e confirmar que nenhuma variável proibida está na lista.

In [3]:
# Variáveis PROIBIDAS (pós-evento — leakage)
VARIAVEIS_PROIBIDAS = [
    'status_cobranca', 'status_financeiro', 'status_pedido',
    'saldo_vencido', 'quantidade_parcelas_vencidas', 'recebido',
    'primeiro_vencimento_em_atraso', 'dias_em_atraso', 'pdd',
    'saldo_vencido_com_juros', 'total_pago_com_juros',
    'aguardando_pagamento_sem_juros', 'vencidos_sem_juros',
    'recebido_sem_juros_tmb', 'data_quitacao',
    # Identificadores pessoais sem poder preditivo
    'CPF', 'documento', 'documento2', 'nome', 'email', 'telefone_ativo',
    # ID — só serve de índice
    'pedido_id'
]

def auditar_variaveis(colunas: list) -> list:
    """Retorna lista de colunas aprovadas (sem proibidas nem pedido_id)."""
    proibidas_encontradas = [c for c in colunas if c in VARIAVEIS_PROIBIDAS]
    aprovadas = [c for c in colunas if c not in VARIAVEIS_PROIBIDAS]
    if proibidas_encontradas:
        print(f'⚠️  REMOVIDAS ({len(proibidas_encontradas)}): {proibidas_encontradas}')
    else:
        print('✓ Nenhuma variável proibida encontrada.')
    print(f'✓ Variáveis aprovadas: {len(aprovadas)}')
    return aprovadas

print('Função auditar_variaveis() pronta — aguardando lista de Dev 2.')

Função auditar_variaveis() pronta — aguardando lista de Dev 2.


---
## FASE 2 — Análise Exploratória (EDA)
> **SUPORTE** — calcular taxa de FPD e peso de classe para os modelos.

In [4]:
# ── Carregar dados (aguardar df_clean de Dev 2) ───────────────────────────────
# Leitura provisória para EDA inicial
df_raw = pd.read_excel('base-treinamento.xlsx', engine='openpyxl')
print(f'Base carregada: {df_raw.shape[0]:,} linhas x {df_raw.shape[1]} colunas')
print(f'\nColunas: {list(df_raw.columns)}')

KeyboardInterrupt: 

In [ ]:
# ── Taxa de FPD e peso de classe ──────────────────────────────────────────────
TARGET = 'fpd_target'

contagem = df_raw[TARGET].value_counts()
n_total  = len(df_raw)
n_fpd    = contagem.get(1, 0)
n_nao_fpd = contagem.get(0, 0)
taxa_fpd = n_fpd / n_total

# Peso de classe para XGBoost (scale_pos_weight = nao_fpd / fpd)
SCALE_POS_WEIGHT = n_nao_fpd / n_fpd

print(f'Total de registros : {n_total:,}')
print(f'FPD = 1            : {n_fpd:,} ({taxa_fpd:.1%})')
print(f'FPD = 0            : {n_nao_fpd:,} ({1-taxa_fpd:.1%})')
print(f'scale_pos_weight   : {SCALE_POS_WEIGHT:.2f}  (usar no XGBoost)')

# Class weights para sklearn / LightGBM
CLASS_WEIGHT = {0: 1.0, 1: SCALE_POS_WEIGHT}
print(f'class_weight dict  : {CLASS_WEIGHT}')

---
## FASE 5 — Baseline: Regressão Logística
> **LÍDER** — treinar após receber X_train / X_val de Dev 3.

In [ ]:
def treinar_baseline(X_train, y_train, X_val, y_val):
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_train)
    X_vl_s = scaler.transform(X_val)

    lr = LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        random_state=RANDOM_STATE
    )
    lr.fit(X_tr_s, y_train)

    prob_val = lr.predict_proba(X_vl_s)[:, 1]
    auc = roc_auc_score(y_val, prob_val)
    print(f'[Baseline LR] AUC val = {auc:.4f}  (meta: > 0.60)')
    if auc < 0.55:
        print('⚠️  AUC abaixo de 0.55 — sinalizar para Dev 2 revisar variáveis!')
    return lr, scaler, prob_val, auc

print('Função treinar_baseline() pronta — aguardando splits de Dev 3.')

---
## FASE 6 — Modelos Avançados
> **LÍDER** — LightGBM, XGBoost, CatBoost

In [ ]:
def treinar_lgbm(X_train, y_train, X_val, y_val, cat_features=None):
    params = dict(
        objective='binary',
        metric='auc',
        n_estimators=1000,
        learning_rate=0.05,
        num_leaves=63,
        scale_pos_weight=SCALE_POS_WEIGHT,
        random_state=RANDOM_STATE,
        verbose=-1
    )
    model = lgb.LGBMClassifier(**params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)]
    )
    prob_val = model.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, prob_val)
    print(f'[LightGBM] best iter={model.best_iteration_}  AUC val={auc:.4f}')
    return model, prob_val, auc


def treinar_xgb(X_train, y_train, X_val, y_val):
    model = xgb.XGBClassifier(
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=6,
        scale_pos_weight=SCALE_POS_WEIGHT,
        eval_metric='auc',
        early_stopping_rounds=50,
        random_state=RANDOM_STATE,
        verbosity=0
    )
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    prob_val = model.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, prob_val)
    print(f'[XGBoost]   best iter={model.best_iteration}  AUC val={auc:.4f}')
    return model, prob_val, auc


def treinar_catboost(X_train, y_train, X_val, y_val, cat_features=None):
    model = CatBoostClassifier(
        iterations=1000,
        learning_rate=0.05,
        depth=6,
        scale_pos_weight=SCALE_POS_WEIGHT,
        eval_metric='AUC',
        early_stopping_rounds=50,
        cat_features=cat_features or [],
        random_seed=RANDOM_STATE,
        verbose=0
    )
    model.fit(X_train, y_train, eval_set=(X_val, y_val))
    prob_val = model.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, prob_val)
    print(f'[CatBoost]  best iter={model.best_iteration_}  AUC val={auc:.4f}')
    return model, prob_val, auc

print('Funções treinar_lgbm(), treinar_xgb(), treinar_catboost() prontas.')

---
## FASE 8 — Otimização de Hiperparâmetros (Optuna)
> **LÍDER**

In [ ]:
def otimizar_lgbm(X_train, y_train, X_val, y_val, n_trials=50):
    def objective(trial):
        params = dict(
            objective='binary',
            metric='auc',
            n_estimators=1000,
            learning_rate=trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            num_leaves=trial.suggest_int('num_leaves', 20, 150),
            max_depth=trial.suggest_int('max_depth', 3, 8),
            min_child_samples=trial.suggest_int('min_child_samples', 20, 100),
            subsample=trial.suggest_float('subsample', 0.6, 1.0),
            colsample_bytree=trial.suggest_float('colsample_bytree', 0.6, 1.0),
            reg_alpha=trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
            reg_lambda=trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
            scale_pos_weight=SCALE_POS_WEIGHT,
            random_state=RANDOM_STATE,
            verbose=-1
        )
        m = lgb.LGBMClassifier(**params)
        m.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)]
        )
        return roc_auc_score(y_val, m.predict_proba(X_val)[:, 1])

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=n_trials)
    print(f'Melhor AUC Optuna: {study.best_value:.4f}')
    print(f'Melhores params  : {study.best_params}')
    return study.best_params

print('Função otimizar_lgbm() pronta — rodar pelo menos 50 trials.')

---
## FASE 9 — Ensemble Ponderado
> **LÍDER**

In [ ]:
def ensemble_ponderado(probs_dict: dict, y_val):
    """probs_dict = {'lgbm': (prob_array, auc), 'xgb': (...), 'cat': (...)}"""
    total_auc = sum(auc for _, auc in probs_dict.values())
    pesos = {k: auc / total_auc for k, (_, auc) in probs_dict.items()}
    print('Pesos do ensemble:')
    for k, w in pesos.items():
        print(f'  {k}: {w:.3f}')

    prob_ensemble = sum(pesos[k] * p for k, (p, _) in probs_dict.items())
    auc_ens = roc_auc_score(y_val, prob_ensemble)

    melhor_individual = max(probs_dict.values(), key=lambda x: x[1])[1]
    print(f'\nAUC ensemble      : {auc_ens:.4f}')
    print(f'AUC melhor indiv. : {melhor_individual:.4f}')
    if auc_ens >= melhor_individual:
        print('→ Usar ENSEMBLE como modelo final.')
    else:
        print('→ Usar melhor modelo INDIVIDUAL (ensemble não melhorou).')
    return prob_ensemble, auc_ens, pesos

print('Função ensemble_ponderado() pronta.')

---
## FASE 10 — Calibração de Probabilidades
> **LÍDER**

In [ ]:
def calibrar_modelo(modelo_final, X_val, y_val, metodo='isotonic'):
    calibrado = CalibratedClassifierCV(modelo_final, method=metodo, cv='prefit')
    calibrado.fit(X_val, y_val)
    prob_cal = calibrado.predict_proba(X_val)[:, 1]
    auc_cal  = roc_auc_score(y_val, prob_cal)
    print(f'AUC após calibração ({metodo}): {auc_cal:.4f}')
    return calibrado

print('Função calibrar_modelo() pronta.')

---
## FASE 15 — Inferência e submission.csv
> **LÍDER**

In [ ]:
def gerar_submission(modelo_calibrado, X_test, pedido_ids, threshold, cortes_faixa):
    """
    cortes_faixa = {'Baixo': 0.20, 'Medio': 0.45, 'Alto': 0.70}  # limite superior de cada faixa
    """
    prob_fpd   = modelo_calibrado.predict_proba(X_test)[:, 1]
    classe_fpd = (prob_fpd >= threshold).astype(int)

    def classificar_faixa(p):
        if p <= cortes_faixa['Baixo']:
            return 'Baixo'
        elif p <= cortes_faixa['Medio']:
            return 'Medio'
        elif p <= cortes_faixa['Alto']:
            return 'Alto'
        return 'Critico'

    df_sub = pd.DataFrame({
        'pedido_id' : pedido_ids,
        'prob_fpd'  : prob_fpd,
        'classe_fpd': classe_fpd,
        'faixa_risco': [classificar_faixa(p) for p in prob_fpd]
    })

    out_path = os.path.join(OUTPUTS_DIR, 'submission.csv')
    df_sub.to_csv(out_path, index=False)
    print(f'✓ submission.csv salvo em {out_path}')
    print(df_sub['faixa_risco'].value_counts())
    return df_sub

print('Função gerar_submission() pronta.')